In [1]:
!pip install transformers
!pip install soundfile
!pip install datasets

import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import soundfile as sf
from datasets import load_dataset

INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 6.2 MB/s eta 0:00:00


In [ ]:
model_name = "brunopbb/Fine-Tunning-LABMET"
processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperForConditionalGeneration.from_pretrained(model_name)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.28k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)


def transcribe_audio_from_array(audio_array, sampling_rate):

    inputs = processor(audio_array, return_tensors="pt", sampling_rate=sampling_rate)


    input_features = inputs.input_features.to(device)


    predicted_ids = model.generate(input_features)


    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    return transcription


def transcribe_dataset(dataset_path):

    dataset = load_dataset("brunopbb/ufcg-labmet-fala-texto", split="test")

    transcriptions = []


    for data in dataset:
        audio_array = data['audio']['array']
        sampling_rate = data['audio']['sampling_rate']

        transcription = transcribe_audio_from_array(audio_array, sampling_rate)
        transcriptions.append({"audio_file": data['audio'], "transcription": transcription})

    return transcriptions


dataset_path = "brunopbb/ufcg-labmet-fala-texto"
transcriptions = transcribe_dataset(dataset_path)

for item in transcriptions:
    print(f"Transcrição: {item['transcription']}")


README.md:   0%|          | 0.00/471 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/15.1M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/3.42M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/30 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/9 [00:00<?, ? examples/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Transcrição: ISOELÉTRICA
Transcrição: DUKE
Transcrição: MICROFLORA-RASTREAMENTO
Transcrição: METIL-HIPURICO
Transcrição: FLUDROCORTISONA
Transcrição: BEZAFIBRATO
Transcrição: COPROLOGICO
Transcrição: FUNDOSCOPY
Transcrição: FORMOTEROL
